In [3]:
!pip install torch==2.1.2 torchvision==0.16.2 torchaudio==2.1.2

Defaulting to user installation because normal site-packages is not writeable


In [4]:
import torch
print(torch.__version__)

2.1.2


In [5]:
pip install mmcv==2.1.0 -f https://download.openmmlab.com/mmcv/dist/cu121/torch2.1/index.html

Defaulting to user installation because normal site-packages is not writeable
Looking in links: https://download.openmmlab.com/mmcv/dist/cu121/torch2.1/index.html
Note: you may need to restart the kernel to use updated packages.


In [6]:
!git clone https://github.com/open-mmlab/mmdetection.git

Cloning into 'mmdetection'...
remote: Enumerating objects: 38019, done.
remote: Counting objects: 100% (102/102), done.
remote: Compressing objects: 100% (75/75), done.
remote: Total 38019 (delta 39), reused 60 (delta 25), pack-reused 37917
Receiving objects: 100% (38019/38019), 63.24 MiB | 38.16 MiB/s, done.
Resolving deltas: 100% (26201/26201), done.
Updating files: 100% (2443/2443), done.


In [7]:
!cd mmdetection; python setup.py install

running install
/share/pkg.8/academic-ml/spring-2024/install/spring-2024-pyt/lib/python3.11/site-packages/setuptools/_distutils/cmd.py:66: SetuptoolsDeprecationWarning: setup.py install is deprecated.
!!

        ********************************************************************************
        Please avoid running ``setup.py`` directly.
        Instead, use pypa/build, pypa/installer or other
        standards-based tools.

        See https://blog.ganssle.io/articles/2021/10/setup-py-deprecated.html for details.
        ********************************************************************************

!!
  self.initialize_options()
/share/pkg.8/academic-ml/spring-2024/install/spring-2024-pyt/lib/python3.11/site-packages/setuptools/_distutils/cmd.py:66: EasyInstallDeprecationWarning: easy_install command is deprecated.
!!

        ********************************************************************************
        Please avoid running ``setup.py`` and ``easy_install``.
      

In [8]:
!pip install mmdet

Defaulting to user installation because normal site-packages is not writeable


In [9]:
!git clone https://github.com/yukkyo/voc2coco.git # voc -> cooo

Cloning into 'voc2coco'...
remote: Enumerating objects: 436, done.
remote: Counting objects: 100% (28/28), done.
remote: Compressing objects: 100% (17/17), done.
remote: Total 436 (delta 19), reused 11 (delta 11), pack-reused 408
Receiving objects: 100% (436/436), 214.06 KiB | 3.75 MiB/s, done.
Resolving deltas: 100% (390/390), done.


In [6]:
# Load the configuration file
config_file = './mmdetection/configs/faster_rcnn/faster_rcnn_r50_fpn_1x_coco.py'
checkpoint_file = './mmdetection/checkpoints/faster_rcnn_r50_fpn_1x_coco_20200130-047c8118.pth'

In [7]:
!cd ./mmdetection; mkdir checkpoints
!wget -O ./mmdetection/checkpoints/faster_rcnn_r50_fpn_1x_coco_20200130-047c8118.pth http://download.openmmlab.com/mmdetection/v2.0/faster_rcnn/faster_rcnn_r50_fpn_1x_coco/faster_rcnn_r50_fpn_1x_coco_20200130-047c8118.pth

mkdir: cannot create directory ‘checkpoints’: File exists
--2024-04-15 15:33:55--  http://download.openmmlab.com/mmdetection/v2.0/faster_rcnn/faster_rcnn_r50_fpn_1x_coco/faster_rcnn_r50_fpn_1x_coco_20200130-047c8118.pth
Resolving download.openmmlab.com (download.openmmlab.com)... 47.246.24.170, 47.246.24.251, 47.246.24.171, ...
Connecting to download.openmmlab.com (download.openmmlab.com)|47.246.24.170|:80... connected.
HTTP request sent, awaiting response... 200 OK
Length: 167287506 (160M) [application/octet-stream]
Saving to: ‘./mmdetection/checkpoints/faster_rcnn_r50_fpn_1x_coco_20200130-047c8118.pth’

./mmdetection/check 100%[===================>] 159.54M  25.1MB/s    in 6.9s    

2024-04-15 15:34:02 (23.0 MB/s) - ‘./mmdetection/checkpoints/faster_rcnn_r50_fpn_1x_coco_20200130-047c8118.pth’ saved [167287506/167287506]



In [8]:
# Configuring the dataset, model, and training parameters
from mmengine.config import Config
#from mmcv import Config

In [21]:
import os
os.chdir('/projectnb/ds549/students/qhzhu')  # Python方式改变到上一级目录
print(os.getcwd())  # 打印当前目录确认改变


/projectnb/ds549/students/qhzhu


In [9]:
!pwd

/projectnb/ds549/students/qhzhu


In [10]:
from mmdet.apis import init_detector, inference_detector

config_file = 'blood_detection_upgraded_version/mmdetection/configs/faster_rcnn/faster-rcnn_r50_fpn_1x_coco.py'
checkpoint_file = 'blood_detection_upgraded_version/mmdetection/checkpoints/faster_rcnn_r50_fpn_1x_coco_20200130-047c8118.pth'

In [11]:

cfg = Config.fromfile(config_file)
print(cfg.pretty_text)

auto_scale_lr = dict(base_batch_size=16, enable=False)
backend_args = None
data_root = 'data/coco/'
dataset_type = 'CocoDataset'
default_hooks = dict(
    checkpoint=dict(interval=1, type='CheckpointHook'),
    logger=dict(interval=50, type='LoggerHook'),
    param_scheduler=dict(type='ParamSchedulerHook'),
    sampler_seed=dict(type='DistSamplerSeedHook'),
    timer=dict(type='IterTimerHook'),
    visualization=dict(type='DetVisualizationHook'))
default_scope = 'mmdet'
env_cfg = dict(
    cudnn_benchmark=False,
    dist_cfg=dict(backend='nccl'),
    mp_cfg=dict(mp_start_method='fork', opencv_num_threads=0))
load_from = None
log_level = 'INFO'
log_processor = dict(by_epoch=True, type='LogProcessor', window_size=50)
model = dict(
    backbone=dict(
        depth=50,
        frozen_stages=1,
        init_cfg=dict(checkpoint='torchvision://resnet50', type='Pretrained'),
        norm_cfg=dict(requires_grad=True, type='BN'),
        norm_eval=True,
        num_stages=4,
        out_indices=

In [12]:
!cd checkpoints
!wget -O faster_rcnn_r50_fpn_1x_coco_20200130-047c8118.pth http://download.openmmlab.com/mmdetection/v2.0/faster_rcnn/faster_rcnn_r50_fpn_1x_coco/faster_rcnn_r50_fpn_1x_coco_20200130-047c8118.pth

/bin/bash: line 0: cd: checkpoints: No such file or directory
--2024-04-15 15:34:30--  http://download.openmmlab.com/mmdetection/v2.0/faster_rcnn/faster_rcnn_r50_fpn_1x_coco/faster_rcnn_r50_fpn_1x_coco_20200130-047c8118.pth
Resolving download.openmmlab.com (download.openmmlab.com)... 47.246.24.170, 47.246.24.251, 47.246.24.171, ...
Connecting to download.openmmlab.com (download.openmmlab.com)|47.246.24.170|:80... connected.
HTTP request sent, awaiting response... 200 OK
Length: 167287506 (160M) [application/octet-stream]
Saving to: ‘faster_rcnn_r50_fpn_1x_coco_20200130-047c8118.pth’

faster_rcnn_r50_fpn 100%[===================>] 159.54M  24.5MB/s    in 6.9s    

2024-04-15 15:34:37 (23.2 MB/s) - ‘faster_rcnn_r50_fpn_1x_coco_20200130-047c8118.pth’ saved [167287506/167287506]



In [13]:
cfg.model.roi_head.bbox_head.num_classes = 6 

In [14]:
import mmengine
from mmengine.hooks import Hook
from mmengine.runner import Runner

from mmdet.registry import HOOKS


# Defining a new hook class for saving the best checkpoint based on the evaluation metric.
@HOOKS.register_module()
class SaveBestCheckpointHook(Hook):
    def __init__(self, out_dir, metric='bbox_mAP_50', save_optimizer=True): #  metric='bbox_mAP
        self.out_dir = out_dir
        self.metric = metric
        self.save_optimizer = save_optimizer
        self.best_score = 0.0

    def after_train_epoch(self, runner):
        # Check and save the best model checkpoint after each training epoch based on the evaluation metric.
        if not self.every_n_epochs(runner, 1):
            return
        from mmengine.runner import save_checkpoint
        if runner.log_buffer.output.get(self.metric, 0) > self.best_score:
            self.best_score = runner.log_buffer.output[self.metric]
            save_checkpoint(runner.model, self.out_dir, optimizer=self.save_optimizer)

In [15]:
from mmdet.apis import inference_detector, init_detector
model = init_detector(cfg, checkpoint_file, device='cuda:0')

Loads checkpoint by local backend from path: blood_detection_upgraded_version/mmdetection/checkpoints/faster_rcnn_r50_fpn_1x_coco_20200130-047c8118.pth
The model and loaded state dict do not match exactly

size mismatch for roi_head.bbox_head.fc_cls.weight: copying a param with shape torch.Size([81, 1024]) from checkpoint, the shape in current model is torch.Size([7, 1024]).
size mismatch for roi_head.bbox_head.fc_cls.bias: copying a param with shape torch.Size([81]) from checkpoint, the shape in current model is torch.Size([7]).
size mismatch for roi_head.bbox_head.fc_reg.weight: copying a param with shape torch.Size([320, 1024]) from checkpoint, the shape in current model is torch.Size([24, 1024]).
size mismatch for roi_head.bbox_head.fc_reg.bias: copying a param with shape torch.Size([320]) from checkpoint, the shape in current model is torch.Size([24]).


In [29]:
import os
os.chdir('/projectnb/ds549/students/qhzhu')  # Python方式改变到上一级目录
print(os.getcwd())  # 打印当前目录确认改变


/projectnb/ds549/students/qhzhu


In [16]:
image_path = 'Blood detection dataset/blood/normal_cases/jpg/img_hod_000003.jpg'

In [17]:
import mmcv

In [18]:
import os
import cv2
from mmdet.apis import inference_detector, init_detector

#checking the model file
checkpoint_path = 'blood_detection_upgraded_version/latest.pth'
assert os.path.isfile(checkpoint_path), f"Checkpoint file not found at {checkpoint_path}"

assert os.path.isfile(image_path), f"Image file not found at {image_path}"

#loading the model
model = init_detector(cfg, checkpoint_path, device='cuda:0')

Loads checkpoint by local backend from path: blood_detection_upgraded_version/latest.pth


In [41]:
def blood_classifier(img_path, model, device):
    img = cv2.imread(img_path)
    result = inference_detector(model, img)
    pred_instances = result.pred_instances
    
    CLASSES = ('class1', 'class2', 'blood', 'class4', 'class5', 'class6')
    highest_confidence = 0
    detected_class = None

    # 访问InstanceData中的scores和labels
    scores = pred_instances.scores.cpu().numpy()
    labels = pred_instances.labels.cpu().numpy()

    for score, label in zip(scores, labels):
        if score > highest_confidence:
            highest_confidence = score
            detected_class = CLASSES[label]

    # 根据检测结果分类
    if detected_class == 'blood':
        classification = 'blood'
    else:
        classification = None

    return classification


In [40]:
print(blood_classifier(image_path,model,device='cuda:0'))

blood
